# Figure 4

In [1]:
options(warn=-1)

In [2]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
        
            # Seurat 
            library(Seurat), 
                         
            # Data 
            library(tidyverse), 
            library(data.table), 
            library(reactable), 

            # Plotting 
            library(ggplot2), 
            library(ComplexHeatmap), 
            library(patchwork), 
            library(cowplot), 
            library(ggrepel), 
            library(circlize), 
            library(ggridges), 
            library(paletteer), 
            
            # Parallel 
            library(BiocParallel), 

            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [3]:
# Configure reticulate 
use_condaenv(condaenv="p.3.8.12-FD20220623HSCT", conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)

In [4]:
random_seed <- 42
set.seed(random_seed)

In [5]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20220623HSCT")

In [6]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=4)) # From project global source()
# showtext::showtext_auto() 

In [7]:
# Source scripts 
source("bin/seurat_qc.R")
source("script/figure/bin/dp_feature.R")
source("script/figure/bin/line_plot.R")
source("script/figure/bin/dea_vp.R")
source("script/figure/bin/dea_stats.R")

# Import Seurat object

In [8]:
so <- readRDS("data/object/subsets/annotation/lymphocyte.rds")

# UMAP

In [9]:
dplot_1 <- dplot(so, reduction="umap", group_by="leiden_cell_type_main", alpha=1, pt_size=0.5, size_select=4) +

    ggtitle("Cell type leiden") +
    scale_color_manual(values=color$cell_type_leiden_subset_r, na.value="grey50", labels=base::str2expression(levels(so$leiden_cell_type_main))) + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    theme(
    
        legend.text.align=0
    
    ) 

In [10]:
pdf("result/figures/figure_4/umap_cell_type_leiden_subset_lymphocyte.pdf", width=10, height=3)

dplot_1

dev.off()

png 
  2

In [11]:
fplot_1 <- fplot(so, reduction="umap", features="cc_aa_alignment_size", alpha=1, pt_size=0.5, size_select=4, viridis_option="A") + 
    ggtitle("Clonal expansion") + guides(color=guide_colourbar(title="#Clones", barwidth=unit(2, "mm"), barheight=unit(10, "mm"))) 

In [12]:
pdf("result/figures/figure_4/umap_cc_aa_alignment_size_lymphocyte.pdf", width=10, height=3)

fplot_1

dev.off()

png 
  2

# Marker gene expression 

In [13]:
# Normalize 
so <- NormalizeData(so, assay="RNA", normalization.method="LogNormalize", scale.factor=1e6)

In [14]:
# Genes and categories
genes <- read.csv("script/figure/figure_4/data/figure_4_dotplot_celltype_markers.csv")

In [15]:
dp_1 <- dp_feature(so, genes$gene, group_by="leiden_cell_type_main", group_by_order=rev(levels(so$leiden_cell_type_main)), split=genes$group, split_order=unique(genes$group), range_max=3, scale=TRUE) + 
    scale_y_discrete(labels=base::str2expression(rev(levels(so$leiden_cell_type_main)))) 

`summarise()` has grouped output by 'leiden_cell_type_main'. You can override
using the `.groups` argument.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.


In [16]:
pdf("result/figures/figure_4/dp_feature_leiden_cell_type_subset.pdf", width=nrow(genes)*0.1+2.5, height=1.5+length(unique(genes$group))*0.1)

dp_1

dev.off()

png 
  2

# Enricher 

## Blood T-cells 

In [17]:
source("plotting_global.R")

enricher <- read.csv("script/figure/figure_4/data/figure_4_tcell_blood_enricher.csv") %>% 
    dplyr::mutate(Msig.DB.Hallmark.2020.pathway=expression_convert(Msig.DB.Hallmark.2020.pathway)) %>%
    dplyr::mutate(Msig.DB.Hallmark.2020.pathway=factor(Msig.DB.Hallmark.2020.pathway, levels=unique(Msig.DB.Hallmark.2020.pathway))) %>%
    reshape2::melt(., id.vars=c("Msig.DB.Hallmark.2020.pathway"), variable="cell_type", value.name=c("count"))  %>%
    dplyr::rename(pathway=Msig.DB.Hallmark.2020.pathway) %>%
    dplyr::mutate(cell_type=ifelse(cell_type=="Th.em", grep("h, em/effector", names(color$cell_type_leiden_subset_r), value=TRUE), as.character(cell_type))) %>%
    dplyr::mutate(cell_type=ifelse(cell_type=="Tc.em", grep("c, em/effector", names(color$cell_type_leiden_subset_r), value=TRUE), as.character(cell_type))) %>% 
    dplyr::mutate(cell_type=factor(cell_type, levels=rev(names(color$cell_type_leiden_subset_r)))) %>% droplevels() %>% 
    tidyr::complete(., pathway, cell_type, fill=list(count=0), explicit=TRUE)

In [18]:
labels <- names(color$cell_type_leiden_subset_r)
labels <- labels[labels %in% enricher$cell_type]

bp_1 <- ggplot(enricher, aes(x=pathway, y=log10(count+1), fill=cell_type, group=cell_type)) + 
    geom_bar(stat="identity", position="dodge", color="black", size=0.1) + 
    xlab("") + ylab("log10(count+1)") + 
    scale_fill_manual(values=color$cell_type_leiden_subset_r, labels=parse(text=labels)) +     
    coord_flip() + facet_grid(pathway~., scales="free", space="free") + 
    scale_x_discrete(labels=scales::parse_format()) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title=NULL, reverse=TRUE)) + 
    theme(
        strip.text=element_blank(), 
        axis.text.y=element_text(vjust=0.5, hjust=1)
    
) 

In [19]:
pdf("result/figures/figure_4/bp_tcell_blood_enricher.pdf", width=3, height=1.5+0.05*length(unique(enricher$pathway))*length(unique(enricher$cell_type)))

bp_1

dev.off()

png 
  2

## Skin T-cells

In [20]:
enricher <- read.csv("script/figure/figure_4/data/figure_4_tcell_skin_enricher.csv", ) %>% 
    dplyr::mutate(Msig.DB.Hallmark.2020.pathway=expression_convert(Msig.DB.Hallmark.2020.pathway)) %>%
    dplyr::mutate(Msig.DB.Hallmark.2020.pathway=factor(Msig.DB.Hallmark.2020.pathway, levels=unique(Msig.DB.Hallmark.2020.pathway))) %>%
    reshape2::melt(., id.vars=c("Msig.DB.Hallmark.2020.pathway"), variable="cell_type", value.name=c("count"))  %>%
    dplyr::rename(pathway=Msig.DB.Hallmark.2020.pathway) %>%
    dplyr::mutate(cell_type=ifelse(cell_type=="Th.mem.eff", grep("h, memory/effector", names(color$cell_type_leiden_subset_r), value=TRUE), as.character(cell_type))) %>%
    dplyr::mutate(cell_type=ifelse(cell_type=="Tc.mem.eff", grep("c, memory/effector", names(color$cell_type_leiden_subset_r), value=TRUE), as.character(cell_type))) %>% 
    dplyr::mutate(cell_type=factor(cell_type, levels=rev(names(color$cell_type_leiden_subset_r)))) %>% droplevels() %>%
    tidyr::complete(., pathway, cell_type, fill=list(count=0), explicit=TRUE)

In [21]:
labels <- names(color$cell_type_leiden_subset_r)
labels <- labels[labels %in% enricher$cell_type]

bp_1 <- ggplot(enricher, aes(x=pathway, y=log10(count+1), fill=cell_type, group=cell_type)) + 
    geom_bar(stat="identity", position="dodge", color="black", size=0.1) + 
    xlab("") + ylab("log10(count+1)") + 
    scale_fill_manual(values=color$cell_type_leiden_subset_r, labels=parse(text=labels)) +     
    coord_flip() + facet_grid(pathway~., scales="free", space="free") + 
    scale_x_discrete(labels=scales::parse_format()) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title=NULL, reverse=TRUE)) + 
    theme(
        strip.text=element_blank(), 
        axis.text.y=element_text(vjust=0.5, hjust=1)
    
) 

In [22]:
pdf("result/figures/figure_4/bp_tcell_skin_enricher.pdf", width=3, height=1.5+0.05*length(unique(enricher$pathway))*length(unique(enricher$cell_type)))

bp_1

dev.off()

png 
  2

# DEA Baseline vs GVHD selected genes

## Skin T cells 

In [23]:
genes <- read.csv("script/figure/figure_4/data/figure_4_tcells_skin_heatmap_deg_baseline_vs_gvhd.csv")

In [24]:
grouping_var <- "time_point"
random_effect_var <- "patient_id"
cell_type <- "leiden_cell_type_main"
cell_min <- 10

In [25]:
# Subset Seurat object
so_i <- subset(so, hto_demux_class=="Skin")

In [26]:
# Get Counts 
cnt <- GetAssayData(so_i, assay="RNA", slot="counts")

In [27]:
# Prepare count data for split 
cnt <- t(as.matrix(cnt))
cnt <- as.data.frame(cnt)

In [28]:
# Make pseudobulks by suming single cell counts
cnt <- split(cnt, f=paste0(so_i[[grouping_var, drop=TRUE]], "|", so_i[[random_effect_var, drop=TRUE]], "|", so_i[[cell_type, drop=TRUE]]))

In [29]:
# Filter by min cell count 
cnt <- cnt[sapply(cnt, function(x) nrow(x)>=cell_min)]

In [30]:
cnt <- lapply(names(cnt), function(i) {x <- data.frame(counts=colSums(cnt[[i]])); colnames(x) <- i; return(x)})
cnt <- do.call(cbind, cnt)

In [31]:
# Subset by cell type 
cnt <- cnt[, grepl("h, memory/effector|c, memory/effector", colnames(cnt))]

In [32]:
# Subset by time point
cnt <- cnt[, grepl("Baseline|GVHD", colnames(cnt))]

In [33]:
# Normalize 
mat <- 10^6*sweep(cnt, 2, colSums(cnt),`/`)
mat <- log(mat+1)

In [34]:
# z-score cpm
mat <- t(scale(t(mat), center=TRUE, scale=TRUE))

In [35]:
# Subset by genes 
mat <- mat[genes$gene, ]

In [36]:
# Column order by time points
mat <- mat[, order(factor(sapply(strsplit(colnames(mat), "\\|"), `[[`, 1), levels=names(color$time_point)))]

In [37]:
# Column split
column_split <- sapply(strsplit(colnames(mat), "\\|"), `[[`, 3) 
column_split <- factor(column_split, levels=names(color$cell_type_leiden_subset_r))
column_split <- droplevels(column_split)

In [38]:
# Color function - DEG count
color_ramp_mat <- c(rev(brewer.pal(11,"RdBu"))[1], "#ffffff", rev(brewer.pal(11,"RdBu"))[11])
breaks_mat <- seq(-round(quantile(abs(mat), 0.99)), round(quantile(abs(mat), 0.99)), length.out=length(color_ramp_mat))
color_function_mat <- circlize::colorRamp2(breaks_mat, color_ramp_mat) 

In [39]:
hm_list <- lapply(seq_along(levels(column_split)), function(idx) {
    
    # Subset data
    mat <- mat[, column_split==levels(column_split)[idx]]
    
    # Column split 
    column_split_idx <- factor(sapply(strsplit(colnames(mat), "\\|"), `[[`, 1), levels=names(color$time_point))
    
    # Get row annotation 
    top_annotation <- ComplexHeatmap::HeatmapAnnotation(

        time_point=column_split_idx %>% droplevels(), 


            annotation_legend_param=list(

                time_point=list(title="Time point", title_gp=gpar(fontsize=6, fontface="plain"), labels_gp=gpar(fontsize=6), grid_width=unit(2, "mm"), grid_height=unit(2, "mm"), legend_width=unit(2, "mm"), legend_height=unit(2, "mm"), border="asis")
                
            ), 

            col=list(

                time_point=color$time_point
            ), 

            simple_anno_size=unit(2.5, "mm"), 
            show_annotation_name=FALSE, 
            show_legend=TRUE

    )
    
    hm <- ComplexHeatmap::Heatmap(
    
            matrix=mat,

            col=color_function_mat, 
            na_col="white", 

            width=unit(1.25*ncol(mat), "mm"), 
            height=unit(2*nrow(mat), "mm"), 

            row_title=NULL, 
            row_title_gp=gpar(fontsize=8, fontface="bold"), 

            column_title=parse(text=levels(column_split)[idx]), 
            column_title_gp=gpar(fontsize=8, fontface="bold"), 

            row_names_gp=gpar(fontsize=6, fontface="italic"), 
            column_names_gp=gpar(fontsize=8, fontface="plain"), 

            cluster_rows=FALSE, 
            show_row_dend=FALSE,
            row_split=genes$regulation, 
            row_gap=unit(2, "mm"), 
            show_row_names=TRUE, 
            row_names_side="left", 

            cluster_columns=TRUE, 
            cluster_column_slices=FALSE, 
            show_column_dend=FALSE, 
            column_split=column_split_idx, 
            column_gap=unit(1, "mm"), 
            show_column_names=FALSE, 

            top_annotation=top_annotation, 
            left_annotation=NULL, 

            rect_gp=gpar(col=NA, lwd=0, alpha=1), 

            heatmap_legend_param=list(title="z-score", title_gp=gpar(fontsize=6, fontface="plain"), labels_gp=gpar(fontsize=6), grid_width=unit(2, "mm"), legend_width=unit(2, "mm"), grid_height=unit(10, "mm"), legend_height=unit(10, "mm"), at=breaks_mat), 
            show_heatmap_legend=ifelse(idx==1, TRUE, FALSE),

            border=TRUE, 
            border_gp=gpar(col="black", lwd=unit(0.2, "mm")), 

            use_raster=FALSE, raster_by_magick=FALSE

    )

}
                 )

hm <- Reduce(`+`, hm_list)

In [40]:
pdf("result/figures/figure_4/hm_tcell_skin_heatmap_deg_baseline_vs_gvhd.pdf", width=4, height=6)

draw(hm, column_title_gp=gpar(fontsize=8, fontface="bold"), merge_legends=TRUE) 

dev.off()

png 
  2

## Blood T cells 

In [41]:
genes <- read.csv("script/figure/figure_4/data/figure_4_tcells_blood_heatmap_deg_baseline_vs_gvhd.csv")

In [42]:
grouping_var <- "time_point"
random_effect_var <- "patient_id"
cell_type <- "leiden_cell_type_main"
cell_min <- 10

In [43]:
# Subset Seurat object
so_i <- subset(so, hto_demux_class=="Blood")

In [44]:
# Get Counts 
cnt <- GetAssayData(so_i, assay="RNA", slot="counts")

In [45]:
# Prepare count data for split 
cnt <- t(as.matrix(cnt))
cnt <- as.data.frame(cnt)

In [46]:
# Make pseudobulks by suming single cell counts
cnt <- split(cnt, f=paste0(so_i[[grouping_var, drop=TRUE]], "|", so_i[[random_effect_var, drop=TRUE]], "|", so_i[[cell_type, drop=TRUE]]))

In [47]:
# Filter by min cell count 
cnt <- cnt[sapply(cnt, function(x) nrow(x)>=cell_min)]

In [48]:
cnt <- lapply(names(cnt), function(i) {x <- data.frame(counts=colSums(cnt[[i]])); colnames(x) <- i; return(x)})
cnt <- do.call(cbind, cnt)

In [49]:
# Subset by cell type 
cnt <- cnt[, grepl("h, em|c, em", colnames(cnt))]

In [50]:
# Subset by time point
cnt <- cnt[, grepl("Baseline|GVHD", colnames(cnt))]

In [51]:
# Normalize 
mat <- 10^6*sweep(cnt, 2, colSums(cnt),`/`)
mat <- log(mat+1)

In [52]:
# z-score cpm
mat <- t(scale(t(mat), center=TRUE, scale=TRUE))

In [53]:
# Subset by genes 
mat <- mat[genes$gene, ]

In [54]:
# Column order by time points
mat <- mat[, order(factor(sapply(strsplit(colnames(mat), "\\|"), `[[`, 1), levels=names(color$time_point)))]

In [55]:
# Column split
column_split <- sapply(strsplit(colnames(mat), "\\|"), `[[`, 3) 
column_split <- factor(column_split, levels=names(color$cell_type_leiden_subset_r))
column_split <- droplevels(column_split)

In [56]:
# Color function - DEG count
color_ramp_mat <- c(rev(brewer.pal(11,"RdBu"))[1], "#ffffff", rev(brewer.pal(11,"RdBu"))[11])
breaks_mat <- seq(-round(quantile(abs(mat), 0.99)), round(quantile(abs(mat), 0.99)), length.out=length(color_ramp_mat))
color_function_mat <- circlize::colorRamp2(breaks_mat, color_ramp_mat) 

In [57]:
hm_list <- lapply(seq_along(levels(column_split)), function(idx) {
    
    # Subset data
    mat <- mat[, column_split==levels(column_split)[idx]]
    
    # Column split 
    column_split_idx <- factor(sapply(strsplit(colnames(mat), "\\|"), `[[`, 1), levels=names(color$time_point))
    
    # Get row annotation 
    top_annotation <- ComplexHeatmap::HeatmapAnnotation(

        time_point=column_split_idx %>% droplevels(), 


            annotation_legend_param=list(

                time_point=list(title="Time point", title_gp=gpar(fontsize=6, fontface="plain"), labels_gp=gpar(fontsize=6), grid_width=unit(2, "mm"), grid_height=unit(2, "mm"), legend_width=unit(2, "mm"), legend_height=unit(2, "mm"), border="asis")
                
            ), 

            col=list(

                time_point=color$time_point
            ), 

            simple_anno_size=unit(2.5, "mm"), 
            show_annotation_name=FALSE, 
            show_legend=TRUE

    )
    
    hm <- ComplexHeatmap::Heatmap(
    
            matrix=mat,

            col=color_function_mat, 
            na_col="white", 

            width=unit(1.25*ncol(mat), "mm"), 
            height=unit(2*nrow(mat), "mm"), 

            row_title=NULL, 
            row_title_gp=gpar(fontsize=8, fontface="bold"), 

            column_title=parse(text=levels(column_split)[idx]), 
            column_title_gp=gpar(fontsize=8, fontface="bold"), 

            row_names_gp=gpar(fontsize=6, fontface="italic"), 
            column_names_gp=gpar(fontsize=8, fontface="plain"), 

            cluster_rows=FALSE, 
            show_row_dend=FALSE,
            row_split=genes$regulation, 
            row_gap=unit(2, "mm"), 
            show_row_names=TRUE, 
            row_names_side="left", 

            cluster_columns=TRUE, 
            cluster_column_slices=FALSE, 
            show_column_dend=FALSE, 
            column_split=column_split_idx, 
            column_gap=unit(1, "mm"), 
            show_column_names=FALSE, 

            top_annotation=top_annotation, 
            left_annotation=NULL, 

            rect_gp=gpar(col=NA, lwd=0, alpha=1), 

            heatmap_legend_param=list(title="z-score", title_gp=gpar(fontsize=6, fontface="plain"), labels_gp=gpar(fontsize=6), grid_width=unit(2, "mm"), legend_width=unit(2, "mm"), grid_height=unit(10, "mm"), legend_height=unit(10, "mm"), at=breaks_mat), 
            show_heatmap_legend=ifelse(idx==1, TRUE, FALSE),

            border=TRUE, 
            border_gp=gpar(col="black", lwd=unit(0.2, "mm")), 

            use_raster=FALSE, raster_by_magick=FALSE

    )

}
                 )

hm <- Reduce(`+`, hm_list)

In [58]:
pdf("result/figures/figure_4/hm_tcell_blood_heatmap_deg_baseline_vs_gvhd.pdf", width=4, height=6)

draw(hm, column_title_gp=gpar(fontsize=8, fontface="bold"), merge_legends=TRUE) 

dev.off()

png 
  2

# Clonality analysis

In [59]:
scirpy <- read.csv("result/scirpy/scirpy.csv", row.names=1) %>% 
     dplyr::mutate(cc_aa_alignment=paste0("CTID_", cc_aa_alignment)) 

In [60]:
so@meta.data <- dplyr::select(so@meta.data, -has_ir, -cc_aa_alignment, -cc_aa_alignment_size)
so <- AddMetaData(so, scirpy)

# T cell clonality over time 

## Number of clonotypes per cell type over time

In [61]:
data <- so@meta.data %>% 
    dplyr::select(patient_id, time_point, hto_demux_class, leiden_cell_type_main, cc_aa_alignment, cc_aa_alignment_size) %>% 
    dplyr::filter(leiden_cell_type_main %in% grep("^T|Cycling", names(color$cell_type_leiden_subset_r), value=TRUE)) %>% 
    na.omit() %>% droplevels() %>% 
    dplyr::select(time_point, hto_demux_class, leiden_cell_type_main, cc_aa_alignment) %>% dplyr::distinct() %>% 
    dplyr::group_by(time_point, hto_demux_class, leiden_cell_type_main) %>% dplyr::summarise(clonotype_n=n())

`summarise()` has grouped output by 'time_point', 'hto_demux_class'. You can
override using the `.groups` argument.


In [62]:
labels <- color$cell_type_leiden_subset_r
labels <- labels[names(labels) %in% levels(data$leiden_cell_type_main)]

bp_1 <- ggplot(data, aes(x=time_point, y=clonotype_n, fill=leiden_cell_type_main)) + 
    geom_bar(stat="identity", color="black", size=0.1) + 
    xlab("") + ylab("count") + 
    facet_grid(~hto_demux_class, scales="free") + 
    scale_fill_manual(values=color$cell_type_leiden_subset_r, labels=parse(text=names(labels))) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title="Cell type", reverse=FALSE)) + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1))

bp_2 <- ggplot(data, aes(x=time_point, y=log(clonotype_n), fill=leiden_cell_type_main)) + 
    geom_bar(stat="identity", color="black", size=0.1) + 
    xlab("") + ylab("Clonotype log(count)") + 
    facet_grid(~hto_demux_class, scales="free") + 
    scale_fill_manual(values=color$cell_type_leiden_subset_r, labels=parse(text=names(labels))) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title="Cell type", reverse=FALSE)) + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1))

In [63]:
pdf("result/figures/figure_4/bp_tcell_clonotype_time_point_count.pdf", width=2.5, height=2)

bp_1 

dev.off()

png 
  2

In [64]:
pdf("result/figures/figure_4/bp_tcell_clonotype_time_point_log_count.pdf", width=2.5, height=2)

bp_2

dev.off()

png 
  2

## Distribution of clones focused on cell abundance 

In [65]:
data <- so@meta.data %>% 
    dplyr::select(patient_id, time_point, hto_demux_class, leiden_cell_type_main, cc_aa_alignment, cc_aa_alignment_size) %>% 
    dplyr::filter(leiden_cell_type_main %in% grep("^T|Cycling", names(color$cell_type_leiden_subset_r), value=TRUE)) %>% 
    na.omit() %>% droplevels() 

In [66]:
data_1 <- data[data$hto_demux_class=="Blood", ]
data_2 <- data[data$hto_demux_class=="Skin", ]

In [67]:
hp <- function(data) {
    
    # Reverse time point levels 
    levels(data$time_point) <- rev(levels(data$time_point))
    
    # Set levels
    leiden_cell_type_main_levels <- levels(data$leiden_cell_type_main)
    
    # Simplify names 
    data$leiden_cell_type_main <- gsub("em/effector", "em", data$leiden_cell_type_main)
    data$leiden_cell_type_main <- gsub("memory/effector", "mem", data$leiden_cell_type_main)
    data$leiden_cell_type_main <- gsub("Cycling", "T['cycling']", data$leiden_cell_type_main)
    
    leiden_cell_type_main_levels <- gsub("em/effector", "em", leiden_cell_type_main_levels)
    leiden_cell_type_main_levels <- gsub("memory/effector", "mem", leiden_cell_type_main_levels)
    leiden_cell_type_main_levels <- gsub("Cycling", "T['cycling']", leiden_cell_type_main_levels)
    
    data$leiden_cell_type_main <- factor(data$leiden_cell_type_main, levels=leiden_cell_type_main_levels)
    
    ggplot(data, aes(y=time_point, x=cc_aa_alignment_size,  fill=time_point)) +
        geom_density_ridges(bandwidth=5, alpha=1, scale=1, rel_min_height=0, color="black", size=0.1) + 
        xlab("") + ylab("") + 
        scale_fill_manual(values=color$time_point) + 
        scale_y_discrete(expand=c(0, 0)) + 
        scale_x_continuous(breaks=c(0, 100)) + 
        facet_grid(~leiden_cell_type_main, labeller=label_parsed, space="free") + 
        guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1), keywidth=0, keyheight=0.3, default.unit="cm", title="Cell type", reverse=TRUE)) + 
        theme(
            
            axis.text.y.left=element_text(hjust=1), 
            strip.text=element_text(size=8), 
            strip.background=element_rect(fill="transparent", color=NA)
        
        )
    
}

In [68]:
hp_1 <- hp(data_1)
hp_2 <- hp(data_2)

In [69]:
pdf("result/figures/figure_4/hp_tcell_clonotype_time_point_blood.pdf", width=1.25+0.5*length(unique(data_1$leiden_cell_type_main)), height=2)

hp_1 

dev.off()

png 
  2

In [70]:
pdf("result/figures/figure_4/hp_tcell_clonotype_time_point_skin.pdf", width=1.25+0.5*length(unique(data_2$leiden_cell_type_main)), height=2)

hp_2

dev.off()

png 
  2

## Bar plots of clones per cell type over time points

In [71]:
data <- so@meta.data %>% 
    dplyr::select(patient_id, time_point, hto_demux_class, leiden_cell_type_main, cc_aa_alignment, cc_aa_alignment_size) %>% 
    dplyr::filter(leiden_cell_type_main %in% grep("^T|Cycling", names(color$cell_type_leiden_subset_r), value=TRUE)) %>% 
    na.omit() %>% droplevels() %>% 
    dplyr::group_by(patient_id, time_point, hto_demux_class, leiden_cell_type_main, cc_aa_alignment) %>% 
    dplyr::mutate(cc_aa_alignment_size=n()) %>% 
    dplyr::ungroup() %>% dplyr::distinct() %>% 
    dplyr::mutate(clonotype_class=ifelse(cc_aa_alignment_size == 1, "Singelton (N=1)", NA)) %>% 
    dplyr::mutate(clonotype_class=ifelse(cc_aa_alignment_size %in% 2:5, "Rare (N=2-5)", clonotype_class)) %>%
    dplyr::mutate(clonotype_class=ifelse(cc_aa_alignment_size %in% 6:20, "Intermediate (N=6-20)", clonotype_class)) %>% 
    dplyr::mutate(clonotype_class=ifelse(cc_aa_alignment_size > 20, "Expanded (N>20)", clonotype_class)) %>%
    dplyr::group_by(time_point, hto_demux_class, leiden_cell_type_main) %>% 
    dplyr::mutate(clonotype_n=n()) %>% 
    dplyr::group_by(clonotype_class, clonotype_n, .add=TRUE) %>% 
    dplyr::summarise(clonotype_class_n=n()) %>% 
    dplyr::group_by(across(setdiff(group_vars(.), 'clonotype_class'))) %>% 
    dplyr::ungroup() %>%
    dplyr::mutate(clonotype_class_ratio=clonotype_class_n/clonotype_n) %>% 
    dplyr::mutate(clonotype_class_ratio=ifelse(is.na(clonotype_class_ratio), 0, clonotype_class_ratio)) %>% 
    dplyr::mutate(clonotype_class=factor(clonotype_class, levels=rev(c("Singelton (N=1)", "Rare (N=2-5)", "Intermediate (N=6-20)", "Expanded (N>20)"))))

`summarise()` has grouped output by 'time_point', 'hto_demux_class',
'leiden_cell_type_main', 'clonotype_class'. You can override using the
`.groups` argument.


In [72]:
data_1 <- dplyr::filter(data, hto_demux_class=="Blood") %>% droplevels()
data_2 <- dplyr::filter(data, hto_demux_class=="Skin") %>% droplevels()

In [73]:
bp_tcr <- function(data) {
    
    leiden_cell_type_main_levels <- levels(data$leiden_cell_type_main)
    
    # Simplify names 
    data$leiden_cell_type_main <- gsub("em/effector", "em", data$leiden_cell_type_main)
    data$leiden_cell_type_main <- gsub("memory/effector", "mem", data$leiden_cell_type_main)
    data$leiden_cell_type_main <- gsub("Cycling", "T['cycling']", data$leiden_cell_type_main)
    
    leiden_cell_type_main_levels <- gsub("em/effector", "em", leiden_cell_type_main_levels)
    leiden_cell_type_main_levels <- gsub("memory/effector", "mem", leiden_cell_type_main_levels)
    leiden_cell_type_main_levels <- gsub("Cycling", "T['cycling']", leiden_cell_type_main_levels)
    
    data$leiden_cell_type_main <- factor(data$leiden_cell_type_main, levels=leiden_cell_type_main_levels)
        
    custom_labeller <- function(x) {
                
        return(parse(text=x))
        
    }
    
    p <- ggplot(data, aes(x=time_point, y=clonotype_class_ratio, fill=clonotype_class)) +
        
            geom_bar(stat="identity", color="black", size=0.1) + 
            scale_fill_brewer(palette="Blues") + 
            xlab("") + ylab("Clonotype ratio") +
            scale_y_continuous(expand=c(0.01, 0.01)) + 
            facet_grid(~leiden_cell_type_main, labeller=labeller(leiden_cell_type_main=label_parsed)) +
    
            theme(

                axis.text.x=element_text(angle=90, vjust=0.5, hjust=1), 
                strip.text=element_text(size=8), 
                strip.background=element_rect(fill="transparent", color=NA)

                
            ) + 

            guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1), keywidth=0, keyheight=0.3, default.unit="cm", title="Clonality", reverse=FALSE)) 

    
    return(p)
    
}

In [74]:
p_1 <- bp_tcr(data_1)
p_2 <- bp_tcr(data_2)

In [75]:
pdf("result/figures/figure_4/bp_all_tcr_time_point_blood.pdf", width=1.5+0.5*length(levels(data_1$leiden_cell_type_main)), height=2)

p_1

dev.off()

png 
  2

In [76]:
pdf("result/figures/figure_4/bp_all_tcr_time_point_skin.pdf", width=1.5+0.5*length(levels(data_2$leiden_cell_type_main)), height=2)

p_2

dev.off()

png 
  2

## Clonotype species specificity 

In [77]:
data <- so@meta.data %>% 
    
    # Subset data 
    dplyr::select(genotype_class, time_point, hto_demux_class, leiden_cell_type_main, cc_aa_alignment, cc_aa_alignment_size, species_vdjdb) %>% 
    dplyr::filter(leiden_cell_type_main %in% grep("^T|Cycling", names(color$cell_type_leiden_subset_r), value=TRUE)) %>% 
    na.omit() %>% droplevels()  

In [78]:
data <- so@meta.data %>% 
    
    # Subset data 
    dplyr::select(genotype_class, time_point, hto_demux_class, leiden_cell_type_main, cc_aa_alignment, cc_aa_alignment_size, species_vdjdb) %>% 
    dplyr::filter(leiden_cell_type_main %in% grep("^T|Cycling", names(color$cell_type_leiden_subset_r), value=TRUE)) %>% 
    na.omit() %>% droplevels() %>% 

    # Set time_point 
    dplyr::mutate(time_point=ifelse(time_point=="GVHD", "GVHD", "HSCT")) %>% 
    dplyr::mutate(time_point=factor(time_point, levels=c("HSCT", "GVHD"))) %>% 
    
    # Recompute clonality for respective groups
    dplyr::group_by(hto_demux_class, time_point, cc_aa_alignment) %>% dplyr::mutate(clonotype_n=n()) %>% 
    
    # Annotate clonality class
    dplyr::mutate(clonotype_class=ifelse(clonotype_n == 1, "Singelton", NA)) %>% 
    dplyr::mutate(clonotype_class=ifelse(clonotype_n %in% 2:5, "Rare", clonotype_class)) %>%
    dplyr::mutate(clonotype_class=ifelse(clonotype_n %in% 6:20, "Intermediate", clonotype_class)) %>% 
    dplyr::mutate(clonotype_class=ifelse(clonotype_n > 20, "Expanded", clonotype_class)) %>%
    dplyr::mutate(clonotype_class=factor(clonotype_class, levels=rev(c("Singelton", "Rare", "Intermediate", "Expanded")))) %>% 

    # Fix missing tcr annotations 
    dplyr::mutate(species_vdjdb=ifelse(species_vdjdb=="", "Unknown", species_vdjdb)) %>% 

    # Genotype class ratio 
    dplyr::group_by(clonotype_class, species_vdjdb, clonotype_n, .add=TRUE) %>% dplyr::mutate(clonotype_n=n()) %>% 
    dplyr::summarise(genotype_host_n=sum(genotype_class=="Host"), genotype_donor_n=sum(genotype_class=="Donor")) %>%
    dplyr::mutate(genotype_class=ifelse(genotype_host_n>0 & genotype_donor_n==0, "Host", NA)) %>% 
    dplyr::mutate(genotype_class=ifelse(genotype_host_n==0 & genotype_donor_n>0, "Donor", genotype_class)) %>% 
    dplyr::mutate(genotype_class=ifelse(genotype_host_n>0 & genotype_donor_n>0, "Shared", genotype_class)) %>% 
    
    # Compute ratios 
    dplyr::group_by(across(setdiff(group_vars(.), 'cc_aa_alignment'))) %>% 
    dplyr::group_by(across(setdiff(group_vars(.), 'species_vdjdb'))) %>% 
    dplyr::group_by(genotype_class, .add=TRUE) %>% dplyr::mutate(n=n()) %>% 
    dplyr::group_by(n, species_vdjdb, .add=TRUE) %>% dplyr::summarise(tcr_class_n=n()) %>% dplyr::ungroup() %>% 
    dplyr::mutate(tcr_class_ratio=tcr_class_n/n)

`summarise()` has grouped output by 'hto_demux_class', 'time_point',
'cc_aa_alignment', 'clonotype_class', 'species_vdjdb'. You can override using
the `.groups` argument.
`summarise()` has grouped output by 'hto_demux_class', 'time_point',
'clonotype_class', 'genotype_class', 'n'. You can override using the `.groups`
argument.


In [79]:
species_vdjdb <- factor(data$species_vdjdb)
species_vdjdb <- relevel(species_vdjdb, "Unknown")
species_vdjdb <- levels(species_vdjdb)

In [80]:
color_species_vdjdb <- setNames(c("light gray", "gray", as.character(paletteer_d("palettetown::combusken"))[1:12]), species_vdjdb)

In [81]:
dp <- function(data, title) {
    
    # Set species vdjdb factor levels to full level from color
    data$species_vdjdb <- factor(data$species_vdjdb, levels=names(color_species_vdjdb))
    
    p <- ggplot(data, aes(x=1, ymin=0, ymax=1, y=tcr_class_ratio, fill=species_vdjdb, xmin=0, xmax=1)) +
    
            geom_bar(stat="identity", color="black", size=0.1) + 
            coord_polar(theta="y", start=0) + 
            scale_fill_manual(values=color_species_vdjdb, drop=FALSE) + 
            ggtitle(title) + 
            facet_grid(clonotype_class~time_point) +

            theme(

                axis.text.x=element_blank(), 
                axis.text.y=element_blank(), 
                axis.title.x=element_blank(), 
                axis.title.y=element_blank(), 
                axis.ticks=element_blank(), 
                axis.line=element_blank()

            ) + 

            guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1), keywidth=0, keyheight=0.3, default.unit="cm", title=NULL, reverse=TRUE)) + 
            geom_text(aes(x=0, y=0, label=n), size=2, show.legend=FALSE)
        
    return(p)

}

In [82]:
dp_1 <- dp(data[data$hto_demux_class=="Blood" & data$genotype_class=="Host", ], title="Blood (Host)")
dp_2 <- dp(data[data$hto_demux_class=="Blood" & data$genotype_class=="Donor", ], title="Blood (Donor)")
dp_3 <- dp(data[data$hto_demux_class=="Skin" & data$genotype_class=="Host", ], title="Skin (Host)")
dp_4 <- dp(data[data$hto_demux_class=="Skin" & data$genotype_class=="Donor", ], title="Skin (Donor)")

In [83]:
pdf("result/figures/figure_4/dp_tcr_species_vdj_blood_host.pdf", width=4, height=3)

dp_1

dev.off()

png 
  2

In [84]:
pdf("result/figures/figure_4/dp_tcr_species_vdj_blood_donor.pdf", width=4, height=3)
dp_2

dev.off()

png 
  2

In [85]:
pdf("result/figures/figure_4/dp_tcr_species_vdj_skin_host.pdf", width=4, height=3)
dp_3

dev.off()

png 
  2

In [86]:
pdf("result/figures/figure_4/dp_tcr_species_vdj_skin_donor.pdf", width=4, height=3)
dp_4

dev.off()

png 
  2

## Donut plot shared clones between blood and skin per patient id

In [87]:
data <- so@meta.data %>% 
    dplyr::filter(leiden_cell_type_main %in% grep("^T|Cycling", names(color$cell_type_leiden_subset_r), value=TRUE)) %>% 
    dplyr::filter(time_point %in% c("D14", "D100", "GVHD")) %>% 
    droplevels() %>% 
    dplyr::select(patient_id, cc_aa_alignment, hto_demux_class) %>% na.omit() %>% dplyr::distinct() %>% 
    dplyr::group_by(patient_id, cc_aa_alignment) %>% dplyr::summarise(clonotype_class=n()) %>% 
    dplyr::mutate(clonotype_class=ifelse(clonotype_class==1, "private", "shared")) %>% 
    dplyr::group_by(across(setdiff(group_vars(.), 'cc_aa_alignment'))) %>% 
    dplyr::mutate(clonotype_n=n()) %>%
    dplyr::group_by(clonotype_class, clonotype_n, .add=TRUE) %>% dplyr::summarise(clonotype_class_n=n()) %>% dplyr::ungroup() %>% 
    tidyr::complete(., patient_id, clonotype_class, fill=list(clonotype_class_n=0), explicit=TRUE) %>% 
    dplyr::mutate(clonotype_class_ratio=clonotype_class_n/clonotype_n) %>% 
    dplyr::mutate(clonotype_class_ratio=ifelse(is.na(clonotype_class_ratio), 0, clonotype_class_ratio)) %>% 
    dplyr::mutate(patient_id=gsub("_", " ", patient_id)) %>% 
    dplyr::mutate(patient_id=factor(patient_id, levels=paste("Patient", 1:11))) %>% droplevels()

`summarise()` has grouped output by 'patient_id'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'patient_id', 'clonotype_class'. You can
override using the `.groups` argument.


In [88]:
dp <- ggplot(data, aes(x=2, y=clonotype_class_ratio, fill=clonotype_class)) +
        
        geom_bar(stat="identity", color="white") + 
        
        coord_polar(theta="y", start=0) + 
        scale_fill_brewer(palette="Blues") + 
        
        facet_wrap(~patient_id, nrow=2) +
                
        theme(
        
            axis.text.x=element_blank(), 
            axis.text.y=element_blank(), 
            axis.title.x=element_blank(), 
            axis.title.y=element_blank(), 
            axis.ticks=element_blank(), 
            axis.line=element_blank()
        
        ) +

        guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title=NULL, reverse=TRUE)) + 

        geomtextpath::geom_textpath(
            data=data, 
            position=position_stack(vjust=0.25),
            aes(x=2, y=clonotype_class_ratio, label=paste0("N=", clonotype_class_n)),
            size=2, 
            show.legend=FALSE
        ) 

In [89]:
pdf("result/figures/figure_4/pie_chart_shared_tcr_patient_id.pdf", width=5, height=3)

dp

dev.off()

png 
  2

## Flow plot of blood and skin shared cell types

In [90]:
data <- so@meta.data %>% 
    dplyr::filter(leiden_cell_type_main %in% grep("^T|Cycling", names(color$cell_type_leiden_subset_r), value=TRUE)) %>% 
    dplyr::filter(time_point %in% c("D14", "D100", "GVHD")) %>% 
    droplevels() %>% 
    dplyr::select(cell_id, patient_id, cc_aa_alignment, hto_demux_class, leiden_cell_type_main, time_point) %>% na.omit() %>% 
    dplyr::group_by(patient_id, cc_aa_alignment) %>% 
    dplyr::mutate(clonotype_class=ifelse(length(unique(hto_demux_class))==1, "private", "shared")) %>% 
    dplyr::filter(clonotype_class=="shared") %>% 
    dplyr::mutate(time_point=ifelse(time_point=="GVHD", "GVHD", "HSCT")) %>% droplevels() %>% dplyr::ungroup()

In [91]:
data_1 <- data %>%
    dplyr::filter(time_point=="HSCT") %>% droplevels() %>% 
    dplyr::select(cc_aa_alignment, hto_demux_class, leiden_cell_type_main, time_point) %>% dplyr::distinct() %>% 
    dplyr::group_by(cc_aa_alignment, hto_demux_class, time_point) %>% dplyr::mutate(n=length(unique(leiden_cell_type_main))) %>% dplyr::ungroup() %>% 
    dplyr::filter(n==1) %>% 
    dplyr::group_by(cc_aa_alignment) %>% dplyr::mutate(n=n()) %>% dplyr::ungroup() %>% 
    dplyr::filter(n==2) %>% 
    dplyr::mutate(leiden_cell_type_main=paste0(leiden_cell_type_main, "_", hto_demux_class)) %>% 
    dcast(., cc_aa_alignment~hto_demux_class, value.var="leiden_cell_type_main") %>% 
    group_by(Blood, Skin) %>% summarise(n=n()) %>% tidyr::spread(key=Skin, value=n, fill=0) %>% 
    tibble::column_to_rownames("Blood") %>% as.matrix()

`summarise()` has grouped output by 'Blood'. You can override using the
`.groups` argument.


In [92]:
data_2 <- data %>%
    dplyr::filter(time_point=="GVHD") %>% droplevels() %>% 
    dplyr::select(cc_aa_alignment, hto_demux_class, leiden_cell_type_main, time_point) %>% dplyr::distinct() %>% 
    dplyr::group_by(cc_aa_alignment, hto_demux_class, time_point) %>% dplyr::mutate(n=length(unique(leiden_cell_type_main))) %>% dplyr::ungroup() %>% 
    dplyr::filter(n==1) %>% 
    dplyr::group_by(cc_aa_alignment) %>% dplyr::mutate(n=n()) %>% dplyr::ungroup() %>% 
    dplyr::filter(n==2) %>% 
    dplyr::mutate(leiden_cell_type_main=paste0(leiden_cell_type_main, "_", hto_demux_class)) %>% 
    dcast(., cc_aa_alignment~hto_demux_class, value.var="leiden_cell_type_main") %>% 
    group_by(Blood, Skin) %>% summarise(n=n()) %>% tidyr::spread(key=Skin, value=n, fill=0) %>% 
    tibble::column_to_rownames("Blood") %>% as.matrix()

`summarise()` has grouped output by 'Blood'. You can override using the
`.groups` argument.


In [93]:
chord_diagram <- function(data) {
    
    # Suppress the plot display
    pdf(NULL)
    dev.control(displaylist="enable")
    
    # Set grid colors
    color_1 <- color$cell_type_leiden_subset_r
    color_2 <- color$cell_type_leiden_subset_r

    names(color_1) <- paste0(names(color_1), "_Blood")
    names(color_2) <- paste0(names(color_2), "_Skin")

    grid_col <- c(color_1, color_2)
    grid_col <- grid_col[names(grid_col) %in% c(colnames(data), rownames(data))]
    
    # Set short cell type names 
    rownames(data) <- gsub("em/effector", "em", rownames(data))
    colnames(data) <- gsub("em/effector", "em", colnames(data))
    rownames(data) <- gsub("memory/effector", "mem", rownames(data))
    colnames(data) <- gsub("memory/effector", "mem", colnames(data))
    rownames(data) <- gsub("Cycling", "T['cycling']", rownames(data))
    colnames(data) <- gsub("Cycling", "T['cycling']", colnames(data))
        
    names(grid_col) <- gsub("em/effector", "em", names(grid_col))
    names(grid_col) <- gsub("memory/effector", "mem", names(grid_col))
    names(grid_col) <- gsub("Cycling", "T['cycling']", names(grid_col))

    # Set sector index labels
    labels <- setNames(gsub("_Blood|_Skin", "", names(grid_col)), names(grid_col))

    # Create the chord diagram
    chordDiagram(data, grid.col=grid_col, annotationTrack=c("grid"), annotationTrackHeight=mm_h(10))
    
    abline(h=0, lty=2, col="#000000")
    
    # Set Axis 
    for(sector_index in get.all.sector.index()) {
        
        circos.xaxis(sector.index=sector_index, minor.ticks=0, labels.cex=0.25, labels=FALSE)
        
    }
    
    # Set labels 
    circos.trackPlotRegion(track.index=1, panel.fun=function(x, y) {
        
        # Get sector labels and replace 
        sector.name <- get.cell.meta.data("sector.index")
        labels <- labels[match(sector.name, names(labels))]
        
        circos.text(x=CELL_META$xcenter, y=CELL_META$ylim[2], labels=parse(text=labels), cex=0.5, niceFacing=TRUE, facing="outside", adj=c(0.5, 1.5))

    }, bg.border=NA
                          )
    
    # Record plot
    p <- recordPlot()
    
    # Close the null device
    invisible(dev.off())
    
    return(p)
    
}

In [94]:
plot_1 <- chord_diagram(data_1)
plot_2 <- chord_diagram(data_2)

In [95]:
pdf("result/figures/figure_4/choard_diagram_shared_tcr_hsct.pdf", width=1.5, height=1.5)

replayPlot(plot_1)

dev.off()

png 
  2

In [96]:
pdf("result/figures/figure_4/choard_diagram_shared_tcr_gvhd.pdf", width=1.5, height=1.5)

replayPlot(plot_2)

dev.off()

png 
  2

## Skin vs Blood TCR clone DEA 

In [97]:
feature_select <- function(so, cnt_min, cell_min, grouping_var, random_effect_var, feature_group_min, verbose=TRUE) {
    
    # Select Seurat components 
    meta <- so@meta.data
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
    
    # Prepare data for split 
    cnt <- t(as.matrix(cnt))
    cnt <- as.data.frame(cnt)
    
    # Split expression matrix by grouping variable into list
    cnt <- split(cnt, f=paste(so[[grouping_var, drop=TRUE]], so[[random_effect_var, drop=TRUE]]))
    
    # Check within group expression 
    cnt <- lapply(cnt, function(x) {colSums(x>=cnt_min)>=cell_min})
    cnt <- do.call(rbind, cnt)
    
    # Check across group expression 
    cnt_check <- colSums(cnt)>=feature_group_min
    
    # Get genes 
    genes <- colnames(cnt)[cnt_check]

    # Create Seurat object with gene subset 
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
    cnt <- cnt[genes, , drop=FALSE]

    so <- CreateSeuratObject(counts=cnt, meta=meta)
    
    if(verbose) {message(paste("Final number of genes for testing", nrow(so)))}

    return(so)
    
}

In [98]:
voom_lm_fit <- function(so, grouping_var, random_effect_var, sample_weights, block=NULL) {

        
    # Get Counts 
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
        
    # Prepare count data for split 
    cnt <- as.matrix(cnt)

    # Get grouping variables from cnt matrix
    grouping_var_vec <- so[[grouping_var, drop=TRUE]] # Used for design matrix
    random_effect_var_vec <- so[[random_effect_var, drop=TRUE]] # Used as blocking variable
    
    grouping_var_vec <- as.character(grouping_var_vec) 
    random_effect_var_vec <- as.character(random_effect_var_vec)
        
    design <- stats::model.matrix(~0 + grouping_var_vec + random_effect_var_vec)
    colnames(design) <- gsub("grouping_var_vec|random_effect_var_vec", "", colnames(design))
    
    # Run voomLmFit
    fit <- edgeR::voomLmFit(
        
        counts=cnt, 
        design=design, 
        block=block, 
        normalize.method="none", 
        lib.size=NULL, 
        sample.weights=sample_weights, 
        plot=FALSE

    )
    
    return(fit)
    
}

In [99]:
ebayes <- function(fit, so, grouping_var, grouping_var_vec) {
    
    contrasts_vec <- paste0(grouping_var_vec[1], "-", grouping_var_vec[2])
    contrasts <- limma::makeContrasts(contrasts=contrasts_vec, levels=fit$design)
    contrasts_fit <- limma::contrasts.fit(fit, contrasts=contrasts)
    
    # eBayes fit 
    efit <- limma::eBayes(contrasts_fit)

    # Get result table
    result_df <- limma::topTable(efit, sort.by="P", n=Inf, p.value=1, lfc=0, coef=1)
    
    # Convert results to Seurat format 
    colnames(result_df)[1] <- "avg_log2FC"
    colnames(result_df)[4] <- "p_value"
    colnames(result_df)[5] <- "p_val_adj"
            
    # Add percentage expression 
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
    cnt_bool <- cnt[rownames(result_df), ] > 0

    result_df$pct_1<-rowSums(cnt_bool[, so[[grouping_var, drop=TRUE]]==grouping_var_vec[1]])/sum(so[[grouping_var, drop=TRUE]]==grouping_var_vec[1])
    result_df$pct_2<-rowSums(cnt_bool[, so[[grouping_var, drop=TRUE]]==grouping_var_vec[2]])/sum(so[[grouping_var, drop=TRUE]]==grouping_var_vec[2])
    
    return(result_df)
    
}

In [100]:
grouping_var <- "hto_demux_class"
random_effect_var <- "patient_id"
grouping_var_vec <- c("Skin", "Blood")
sample_weights <- FALSE

cnt_min=1
cell_min=1
feature_group_min=3

p_adj_thr <- 0.05

so_1 <- subset(so, subset=cc_aa_alignment %in% data$cc_aa_alignment)
so_2 <- subset(so, subset=cc_aa_alignment %in% data[data$time_point=="HSCT", ]$cc_aa_alignment)
so_3 <- subset(so, subset=cc_aa_alignment %in% data[data$time_point=="GVHD", ]$cc_aa_alignment)

In [101]:
so_1 <- feature_select(so_1, cnt_min, cell_min, grouping_var, random_effect_var, feature_group_min, verbose=TRUE)
so_2 <- feature_select(so_2, cnt_min, cell_min, grouping_var, random_effect_var, feature_group_min, verbose=TRUE)
so_3 <- feature_select(so_3, cnt_min, cell_min, grouping_var, random_effect_var, feature_group_min, verbose=TRUE)

Final number of genes for testing 6529

Final number of genes for testing 5055

Final number of genes for testing 6253



In [102]:
fit_1 <- voom_lm_fit(so_1, grouping_var=grouping_var, random_effect_var=random_effect_var, sample_weights=sample_weights)
fit_2 <- voom_lm_fit(so_2, grouping_var=grouping_var, random_effect_var=random_effect_var, sample_weights=sample_weights)
fit_3 <- voom_lm_fit(so_3, grouping_var=grouping_var, random_effect_var=random_effect_var, sample_weights=sample_weights)

In [103]:
result_1 <- ebayes(fit_1, so_1, grouping_var=grouping_var, grouping_var_vec=grouping_var_vec)
result_2 <- ebayes(fit_2, so_2, grouping_var=grouping_var, grouping_var_vec=grouping_var_vec)
result_3 <- ebayes(fit_3, so_3, grouping_var=grouping_var, grouping_var_vec=grouping_var_vec)

In [104]:
# Label genes 
label <- read.csv("script/figure/figure_4/data/figure_4_shared_tcr_clonotypes_dea.csv")

In [105]:
vp_1 <- dea_vp(result_2, title="Blood vs Skin TCR clones (HSCT)", color_neg=color$hto_demux_class["Blood"], color_pos=color$hto_demux_class["Skin"], log2fc_thr=0, p_adj_thr=p_adj_thr, top_label=15, parse_title=FALSE, label=label[label$group=="HSCT", ][["gene"]])
vp_2 <- dea_vp(result_3, title="Blood vs Skin TCR clones (GVHD)", color_neg=color$hto_demux_class["Blood"], color_pos=color$hto_demux_class["Skin"], log2fc_thr=0, p_adj_thr=p_adj_thr, top_label=15, parse_title=FALSE, label=label[label$group=="GVHD", ][["gene"]])

In [106]:
pdf("result/figures/figure_4/vp_dea_shared_tcr.pdf", width=6, height=2.5)

vp_1 + vp_2 + patchwork::plot_layout(nrow=1)

dev.off()

png 
  2

# DEA overview lymphocyte cells

## Skin 

In [107]:
results <- readRDS("result/dea/pseudobulk/tx_progression/results_skin.rds")
grouping_stat <- readRDS("result/dea/pseudobulk/tx_progression/grouping_stat_skin.rds")

In [108]:
cell_type_order <- levels(so$leiden_cell_type_main)
cell_type_order <- cell_type_order[cell_type_order %in% names(results)]
p_adj_thr <- 0.05

In [109]:
results <- results[names(results) %in% cell_type_order]
grouping_stat <- grouping_stat[names(results) %in% cell_type_order]

In [110]:
source("script/figure/bin/dea_stats.R")

hm_1 <- grouping_stat_hm(grouping_stat, stat="cell_mean", column_title="Cells", name="log(#Cells)", row_order=cell_type_order)
hm_2 <- grouping_stat_hm(grouping_stat, stat="random_effect_var_n", column_title="Patient ID", name="#Patient", row_order=cell_type_order)
hm_3 <- results_n_hm(results, p_adj_thr=p_adj_thr, column_title="DEG", name="log(#DEG)", row_order=cell_type_order)

In [111]:
pdf("result/figures/figure_4/hm_dea_stat_skin_lymphocyte.pdf", width=3, height=3)

draw(hm_1 + hm_2 + hm_3, heatmap_legend_side="bottom")

dev.off()

png 
  2

## Blood 

In [112]:
results <- readRDS("result/dea/pseudobulk/tx_progression/results_blood.rds")
grouping_stat <- readRDS("result/dea/pseudobulk/tx_progression/grouping_stat_blood.rds")

In [113]:
cell_type_order <- levels(so$leiden_cell_type_main)
cell_type_order <- cell_type_order[cell_type_order %in% names(results)]
p_adj_thr <- 0.05

In [114]:
results <- results[names(results) %in% cell_type_order]
grouping_stat <- grouping_stat[names(results) %in% cell_type_order]

In [115]:
source("script/figure/bin/dea_stats.R")

hm_1 <- grouping_stat_hm(grouping_stat, stat="cell_mean", column_title="Cells", name="log(#Cells)", row_order=cell_type_order)
hm_2 <- grouping_stat_hm(grouping_stat, stat="random_effect_var_n", column_title="Patient ID", name="#Patient", row_order=cell_type_order)
hm_3 <- results_n_hm(results, p_adj_thr=p_adj_thr, column_title="DEG", name="log(#DEG)", row_order=cell_type_order)

In [116]:
pdf("result/figures/figure_4/hm_dea_stat_blood_lymphocyte.pdf", width=3, height=3)

draw(hm_1 + hm_2 + hm_3, heatmap_legend_side="bottom")

dev.off()

png 
  2

# DEA volcano plot 

## Results for pseudobatch lymphocyte blood (PB)

In [117]:
results <- readRDS("result/dea/pseudobulk/tx_progression/results_blood.rds")
results <- results[levels(so$leiden_cell_type_main)]

In [118]:
results <- lapply(results, function(result) result[["GVHD-Baseline"]])
results <- results[!sapply(results, is.null)]

In [119]:
vp_list <- lapply(names(results), function(i) dea_vp(results[[i]], title=i, color_neg=color$time_point["Baseline"], color_pos=color$time_point["GVHD"]))

In [120]:
save_plots_to_pdf(vp_list, nrow=4, ncol=3, file_name="result/figures/figure_4/vp_dea_pb_lymphocyte_blood.pdf")

png 
  2

## Results for pseudobatch lymphocyte skin (PB)

In [121]:
results <- readRDS("result/dea/pseudobulk/tx_progression/results_skin.rds")
results <- results[levels(so$leiden_cell_type_main)]

In [122]:
results <- lapply(results, function(result) result[["GVHD-Baseline"]])
results <- results[!sapply(results, is.null)]

In [123]:
vp_list <- lapply(names(results), function(i) dea_vp(results[[i]], title=i, color_neg=color$time_point["Baseline"], color_pos=color$time_point["GVHD"]))

In [124]:
save_plots_to_pdf(vp_list, nrow=4, ncol=3, file_name="result/figures/figure_4/vp_dea_pb_lymphocyte_skin.pdf")

png 
  2

# Residency score 

In [125]:
so <- readRDS("data/object/pp_subset.rds")

In [126]:
residency_genes <- c("CD69", "PRDM1", "CREM")
recirculating_genes <- c("SELL", "CCR7", "KLF2")

In [127]:
so <- AddModuleScore(so, features=list(residency_genes, recirculating_genes), name=c("TRM_score", "TCM_score"))

In [128]:
# Set baseline score to 0 
so$TRM_score1 <- ifelse(so$TRM_score1<=0, 0, so$TRM_score1)
so$TCM_score2 <- ifelse(so$TCM_score2<=0, 0, so$TCM_score2)

# Min Max normalization 
so$TRM_score1 <- (so$TRM_score1 - min(so$TRM_score1)) / (max(so$TRM_score1) - min(so$TRM_score1))
so$TCM_score2 <- (so$TCM_score2 - min(so$TCM_score2)) / (max(so$TCM_score2) - min(so$TCM_score2))

# Score difference 
so$diffTscore <- so$TRM_score1 - so$TCM_score2

In [129]:
# Save module score for PAGA figure 
write.csv(so@meta.data[, "diffTscore", drop=FALSE], "script/figure/figure_4/data/diffTscore.csv")

In [130]:
p_1 <- ggplot(subset(so, subset=cell_type_leiden_subset %in% c("T['h, memory/effector']", "T['h, em/effector']"))@meta.data, aes(x=genotype_class, y=diffTscore, color=genotype_class)) + 
    geom_violin(size=0.25) +     
    geom_jitter(shape=19, size=0.1, width=0.25) + 
    geom_hline(yintercept=0, linetype="dotted", colour="black") + 
    ggtitle("T cell CD4+") + xlab("") + ylab("Module score diff.") + 
    ylim(-0.3, 0.3) + 
    scale_color_manual(values=color$genotype_class) + 
    facet_grid(~hto_demux_class+time_point) + 
    theme(axis.text.x=element_blank()) + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))

p_2 <- ggplot(subset(so, subset=cell_type_leiden_subset %in% c("T['c, memory/effector']", "T['c, em/effector']"))@meta.data, aes(x=genotype_class, y=diffTscore, color=genotype_class)) + 
    geom_violin(size=0.25) +     
    geom_jitter(shape=19, size=0.1, width=0.25) + 
    geom_hline(yintercept=0, linetype="dotted", colour="black") + 
    ggtitle("T cell CD8+") + xlab("") + ylab("Module score diff.") + 
    ylim(-0.3, 0.3) + 
    scale_color_manual(values=color$genotype_class) + 
    facet_grid(~hto_demux_class+time_point) + 
    theme(axis.text.x=element_blank()) + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))

In [138]:
pdf("result/figures/figure_4/jvp_residency_vs_recirculating_scores.pdf", width=5, height=2)

p_1 + p_2 + patchwork::plot_layout(ncol=2, guides="collect")

dev.off()

png 
  2